<a href="https://colab.research.google.com/github/Chathuranga630/sionna/blob/main/Resilience_Paper_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# ============================================================
# SIONNA 2.1 INSTALLATION FOR GOOGLE COLAB
# ============================================================

!pip install -q --upgrade pip

# Sionna 2.1 / Numba-compatible NumPy
!pip install -q --force-reinstall "numpy==2.2.6"

# Sionna 2.1
!pip install -q "sionna==2.1.0"

print("Installation finished.")
print("NOW RESTART THE RUNTIME:")
print("Runtime -> Restart session")

Installation finished.
NOW RESTART THE RUNTIME:
Runtime -> Restart session


In [14]:
from pathlib import Path
import math
import csv
import numpy as np
import torch
import matplotlib.pyplot as plt

# ============================================================
# Sionna
# ============================================================
from sionna.phy.channel.tr38901 import PanelArray, UMi
from sionna.phy.channel import gen_single_sector_topology

# ============================================================
# Configuration
# ============================================================
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

OUT = Path("r_service_results")
OUT.mkdir(parents=True, exist_ok=True)

# 5G / channel assumptions
FC = 3.5e9                    # Hz
NUM_USERS = 6
NUM_PRBS = 24                 # total available PRBs
PRB_BW = 180e3                # Hz, 1 PRB at 15 kHz SCS
NOISE_POWER = 1e-10           # illustrative normalized receiver noise
BATCH_SIZE = 24               # number of independent channel drops
NUM_TIME_SAMPLES = 1

# Service/slice requirements (illustrative)
# Users 0-2: stringent; users 3-5: relaxed
TARGET_MBPS = np.array([12., 12., 12., 5., 5., 5.])
MIN_MARGIN_MBPS = 1.0

# Recovery/control timing assumptions (illustrative)
TGEN_MEAN_MS = 8.0
TRESP_BASE_MS = 2.0
C_PER_ITER_MS = 0.5
D_MIN_MS = 8.0
T_NMIN_MS = 25.0

# Utility incident threshold
E_TH_MBPS = 4.0

# RIS assumptions (illustrative)
RIS_ITERATIONS = 8
RIS_STEP_MS = 1.5
T_PROC_MS = 7.0
T_SLEEP_MAX_MS = 20.0
RIS_ELEMENTS_PER_STEP = 16

# ============================================================
# Sionna UMi channel
# ============================================================
def build_umi_channel():
    """Create a Sionna 3GPP TR 38.901 UMi uplink channel."""
    bs_array = PanelArray(
        num_rows_per_panel=1,
        num_cols_per_panel=1,
        polarization="single",
        polarization_type="V",
        antenna_pattern="38.901",
        carrier_frequency=FC,
        device=DEVICE,
    )

    ut_array = PanelArray(
        num_rows_per_panel=1,
        num_cols_per_panel=1,
        polarization="single",
        polarization_type="V",
        antenna_pattern="omni",
        carrier_frequency=FC,
        device=DEVICE,
    )

    channel = UMi(
        carrier_frequency=FC,
        o2i_model="low",
        ut_array=ut_array,
        bs_array=bs_array,
        direction="uplink",
        enable_pathloss=True,
        enable_shadow_fading=True,
        device=DEVICE,
        spec_version="19.2",
    )
    return channel


def generate_channel_gains(channel):
    """
    Generate B independent UMi drops for NUM_USERS and return
    a [B, U] mean channel-power gain matrix.
    """
    topology = gen_single_sector_topology(
        batch_size=BATCH_SIZE,
        num_ut=NUM_USERS,
        scenario="umi",
        device=DEVICE,
    )

    channel.set_topology(*topology)

    # h:
    # [batch, num_rx, num_rx_ant, num_tx, num_tx_ant, num_paths, time]
    h, tau = channel(
        num_time_samples=NUM_TIME_SAMPLES,
        sampling_frequency=1e6,
    )

    # Average path power and time.
    power = torch.mean(torch.abs(h) ** 2, dim=(-1, -2))

    # [B, 1, 1, U, 1] -> [B,U]
    gain = power[:, 0, 0, :, 0]

    return gain.detach().cpu().numpy()

# ============================================================
# Rate / allocation model
# ============================================================
def spectral_efficiency(snr_linear):
    """Simple upper-bound spectral efficiency for CPU-friendly study."""
    return np.log2(1.0 + np.maximum(snr_linear, 1e-12))


def user_rate_mbps(channel_gain, prbs):
    """
    Convert Sionna channel gain and assigned PRBs to a rate estimate.
    """
    snr = channel_gain / NOISE_POWER
    se = spectral_efficiency(snr)

    # Cap to a realistic range for this simplified scheduler study.
    se = np.minimum(se, 7.4)

    rate_bps = prbs * PRB_BW * se
    return rate_bps / 1e6


def initial_allocation():
    """
    Priority-weighted starting allocation.
    Stringent users receive larger weights.
    """
    priority = np.array([2.0, 2.0, 2.0, 1.0, 1.0, 1.0])
    raw = priority / priority.sum() * NUM_PRBS
    alloc = np.floor(raw).astype(int)

    while alloc.sum() < NUM_PRBS:
        k = np.argmax(raw - alloc)
        alloc[k] += 1

    return alloc


def recovery_allocation(gain):
    """
    Iteratively increase resources for users below target QoS.
    Returns the complete R_service record for one representative drop.
    """
    alloc = initial_allocation()

    history = []
    cumulative_talloc = 0.0
    cumulative_d = 0.0

    for i in range(1, 31):
        rates = user_rate_mbps(gain, alloc)
        error = rates - TARGET_MBPS
        mse = float(np.mean(error ** 2))

        # Illustrative meta-prompt and response timing.
        tgen = max(1.0, np.random.normal(TGEN_MEAN_MS, 1.0))
        d_i = max(0.5, np.random.normal(3.0, 0.7))
        tresp = d_i + C_PER_ITER_MS
        talloc_i = tgen + tresp

        cumulative_talloc += talloc_i
        cumulative_d += d_i

        history.append({
            "iteration": i,
            "alloc": alloc.copy(),
            "rates": rates.copy(),
            "error": error.copy(),
            "mse": mse,
            "tgen_ms": tgen,
            "d_i_ms": d_i,
            "tresp_ms": tresp,
            "talloc_cum_ms": cumulative_talloc,
            "d_cum_ms": cumulative_d,
        })

        # Recovery condition:
        # expected QoS with a margin must be met for all users.
        satisfied = np.all(rates >= TARGET_MBPS - MIN_MARGIN_MBPS)
        if satisfied:
            break

        # Select the most under-served user.
        deficits = TARGET_MBPS - rates
        k = int(np.argmax(deficits))

        # If all PRBs are already assigned, take one PRB from
        # the least-under-served user with >1 PRB.
        if alloc.sum() < NUM_PRBS:
            alloc[k] += 1
        else:
            donors = np.where(alloc > 1)[0]
            if len(donors) == 0:
                break
            donor = donors[np.argmin(deficits[donors])]
            if donor != k:
                alloc[donor] -= 1
                alloc[k] += 1

    final = history[-1]

    # Incident triage from the paper, with explicit assumptions.
    I1 = final["talloc_cum_ms"] > T_NMIN_MS
    I2 = final["d_cum_ms"] > D_MIN_MS
    I3 = np.max(np.abs(final["error"])) > E_TH_MBPS

    return history, {
        "I1": int(I1),
        "I2": int(I2),
        "I3": int(I3),
        "iterations": len(history),
        "Talloc_ms": final["talloc_cum_ms"],
        "Dsum_ms": final["d_cum_ms"],
        "final_mse": final["mse"],
        "final_rates": final["rates"],
        "final_alloc": final["alloc"],
    }

# ============================================================
# RIS dependency model
# ============================================================
def simulate_ris():
    """
    Section 3.1 RIS dependency model.

    This is a control/reconfiguration abstraction, not a full electromagnetic
    RIS ray-tracing model. It models iterative activation of RIS elements and
    records cumulative reconfiguration time and throughput gain.
    """
    elements = 0
    throughput = 1.0
    rows = []

    for j in range(1, RIS_ITERATIONS + 1):
        elements += RIS_ELEMENTS_PER_STEP

        # Saturating gain curve.
        gain = 1.0 + 1.8 * (1.0 - math.exp(-elements / 80.0))
        throughput = gain

        rows.append({
            "iteration": j,
            "elements_on": elements,
            "tj_ris_ms": RIS_STEP_MS,
            "T_ris_ms": j * RIS_STEP_MS,
            "relative_rate_gain": throughput,
        })

    T_RIS = rows[-1]["T_ris_ms"]
    E_max = max(r["relative_rate_gain"] for r in rows)

    I4 = T_RIS > T_PROC_MS
    I5 = T_SLEEP_MAX_MS < 10.0  # illustrative sleep event condition

    return rows, {
        "T_RIS_ms": T_RIS,
        "E_max": E_max,
        "I4": int(I4),
        "I5": int(I5),
    }

# ============================================================
# R_service aggregation
# ============================================================
def reliability_index(gain_matrix):
    """
    Reliability proxy:
        fraction of user/drop observations satisfying QoS.
    This mirrors the paper's indicator-function idea without inventing
    an exact paper-specific utility equation.
    """
    success = []

    for b in range(gain_matrix.shape[0]):
        alloc = initial_allocation()
        rates = user_rate_mbps(gain_matrix[b], alloc)
        ok = rates >= TARGET_MBPS
        success.append(ok)

    success = np.asarray(success)
    return float(np.mean(success))


def run():
    print("=" * 70)
    print("Sionna R_service simulation")
    print("Device:", DEVICE)
    print("=" * 70)

    print("\n[1] Building Sionna 3GPP TR 38.901 UMi channel...")
    channel = build_umi_channel()

    print("[2] Generating channel realizations...")
    gains = generate_channel_gains(channel)

    # One representative channel drop for detailed recovery analysis.
    rep_gain = gains[0]

    print("[3] Running resource recovery...")
    history, rec = recovery_allocation(rep_gain)

    print("[4] Running RIS dependency model...")
    ris_rows, ris = simulate_ris()

    print("[5] Computing reliability proxy...")
    R_service_phy = reliability_index(gains)

    # Aggregate R_service components.
    # These weights are simulation assumptions, not values from the paper.
    qos_score = 1.0 / (1.0 + rec["final_mse"])
    recovery_score = 1.0 / (1.0 + rec["Talloc_ms"] / 100.0)
    ris_score = 1.0 / (1.0 + ris["T_RIS_ms"] / 100.0)

    w1, w2, w3 = 0.40, 0.35, 0.25
    R_service = (
        w1 * R_service_phy +
        w2 * qos_score +
        w3 * recovery_score
    )

    print("\n========== R_SERVICE RESULTS ==========")
    print(f"PHY/QoS reliability proxy : {R_service_phy:.4f}")
    print(f"Final QoS MSE             : {rec['final_mse']:.4f}")
    print(f"Recovery iterations       : {rec['iterations']}")
    print(f"Talloc                    : {rec['Talloc_ms']:.3f} ms")
    print(f"Sum control delays        : {rec['Dsum_ms']:.3f} ms")
    print(f"RIS T_RIS                 : {ris['T_RIS_ms']:.3f} ms")
    print(f"RIS E_max                 : {ris['E_max']:.3f}")
    print(f"R_service (illustrative)  : {R_service:.4f}")
    print("Incidents I1-I5:",
          rec["I1"], rec["I2"], rec["I3"],
          ris["I4"], ris["I5"])

    # Save scalar results
    with open(OUT / "summary.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["metric", "value"])
        writer.writerow(["R_service", R_service])
        writer.writerow(["PHY_QoS_reliability", R_service_phy])
        writer.writerow(["final_MSE", rec["final_mse"]])
        writer.writerow(["recovery_iterations", rec["iterations"]])
        writer.writerow(["Talloc_ms", rec["Talloc_ms"]])
        writer.writerow(["Dsum_ms", rec["Dsum_ms"]])
        writer.writerow(["T_RIS_ms", ris["T_RIS_ms"]])
        writer.writerow(["E_max", ris["E_max"]])
        for k in ["I1", "I2", "I3"]:
            writer.writerow([k, rec[k]])
        for k in ["I4", "I5"]:
            writer.writerow([k, ris[k]])

    # --------------------------------------------------------
    # Figure 1: QoS recovery
    # --------------------------------------------------------
    it = [r["iteration"] for r in history]
    mse = [r["mse"] for r in history]
    min_rate = [np.min(r["rates"]) for r in history]

    plt.figure(figsize=(8, 5))
    plt.plot(it, min_rate, marker="o", label="Minimum user rate")
    plt.axhline(np.min(TARGET_MBPS), linestyle="--",
                label="Minimum target rate")
    plt.xlabel("Recovery iteration")
    plt.ylabel("Rate (Mbps)")
    plt.title("Sionna-based QoS Recovery")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT / "01_qos_recovery.png", dpi=200)
    plt.close()

    # --------------------------------------------------------
    # Figure 2: MSE
    # --------------------------------------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(it, mse, marker="o")
    plt.xlabel("Recovery iteration")
    plt.ylabel("MSE of QoS deviation (Mbps²)")
    plt.title("QoS Deviation MSE")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUT / "02_qos_mse.png", dpi=200)
    plt.close()

    # --------------------------------------------------------
    # Figure 3: Talloc
    # --------------------------------------------------------
    talloc = [r["talloc_cum_ms"] for r in history]

    plt.figure(figsize=(8, 5))
    plt.plot(it, talloc, marker="o")
    plt.axhline(T_NMIN_MS, linestyle="--", label="Tnmin")
    plt.xlabel("Recovery iteration")
    plt.ylabel("Cumulative allocation time (ms)")
    plt.title("Talloc and Incident I1")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT / "03_talloc_I1.png", dpi=200)
    plt.close()

    # --------------------------------------------------------
    # Figure 4: Control delay
    # --------------------------------------------------------
    dsum = [r["d_cum_ms"] for r in history]

    plt.figure(figsize=(8, 5))
    plt.plot(it, dsum, marker="o")
    plt.axhline(D_MIN_MS, linestyle="--", label="dmin")
    plt.xlabel("Recovery iteration")
    plt.ylabel("Cumulative control delay (ms)")
    plt.title("Control-message Delay and Incident I2")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT / "04_control_delay_I2.png", dpi=200)
    plt.close()

    # --------------------------------------------------------
    # Figure 5: RIS
    # --------------------------------------------------------
    rj = [r["iteration"] for r in ris_rows]
    tr = [r["T_ris_ms"] for r in ris_rows]
    eg = [r["relative_rate_gain"] for r in ris_rows]

    plt.figure(figsize=(8, 5))
    plt.plot(rj, tr, marker="o")
    plt.axhline(T_PROC_MS, linestyle="--", label="Tproc")
    plt.xlabel("RIS reconfiguration iteration")
    plt.ylabel("Cumulative TRIS (ms)")
    plt.title("RIS Reconfiguration Time and Incident I4")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT / "05_ris_TRIS_I4.png", dpi=200)
    plt.close()

    # --------------------------------------------------------
    # Figure 6: Emax
    # --------------------------------------------------------
    plt.figure(figsize=(8, 5))
    plt.plot(rj, eg, marker="o")
    plt.xlabel("RIS iteration")
    plt.ylabel("Relative throughput gain")
    plt.title("RIS Throughput Gain and Emax")
    plt.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.savefig(OUT / "06_ris_Emax.png", dpi=200)
    plt.close()

    # --------------------------------------------------------
    # Figure 7: Per-user final rates
    # --------------------------------------------------------
    final_rates = rec["final_rates"]

    plt.figure(figsize=(8, 5))
    x = np.arange(NUM_USERS)
    width = 0.35
    plt.bar(x - width/2, TARGET_MBPS, width, label="Target")
    plt.bar(x + width/2, final_rates, width, label="Sionna-derived rate")
    plt.xlabel("User")
    plt.ylabel("Rate (Mbps)")
    plt.title("Final Per-user QoS")
    plt.xticks(x, [f"U{k+1}" for k in range(NUM_USERS)])
    plt.grid(True, axis="y", alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUT / "07_final_user_rates.png", dpi=200)
    plt.close()

    # Save detailed recovery CSV
    with open(OUT / "recovery_history.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "iteration", "Talloc_cum_ms", "D_cum_ms",
            "MSE", "Tgen_ms", "Tresp_ms",
            *[f"rate_U{k+1}_Mbps" for k in range(NUM_USERS)],
            *[f"alloc_U{k+1}_PRB" for k in range(NUM_USERS)]
        ])
        for r in history:
            writer.writerow([
                r["iteration"],
                r["talloc_cum_ms"],
                r["d_cum_ms"],
                r["mse"],
                r["tgen_ms"],
                r["tresp_ms"],
                *r["rates"].tolist(),
                *r["alloc"].tolist()
            ])

    with open(OUT / "ris_history.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=ris_rows[0].keys())
        writer.writeheader()
        writer.writerows(ris_rows)

    print(f"\nResults written to: {OUT.resolve()}")


if __name__ == "__main__":
    run()


Sionna R_service simulation
Device: cuda:0

[1] Building Sionna 3GPP TR 38.901 UMi channel...
[2] Generating channel realizations...
[3] Running resource recovery...
[4] Running RIS dependency model...
[5] Computing reliability proxy...

========== R_SERVICE RESULTS ==========
PHY/QoS reliability proxy : 0.0000
Final QoS MSE             : 84.4277
Recovery iterations       : 30
Talloc                    : 337.207 ms
Sum control delays        : 86.532 ms
RIS T_RIS                 : 12.000 ms
RIS E_max                 : 2.437
R_service (illustrative)  : 0.0613
Incidents I1-I5: 1 1 1 1 0

Results written to: /content/r_service_results
